In [ ]:
import pandas as pd 
import numpy as np
df = pd.read_csv("../data/Telco-Customer-Churn.csv")
df.head()


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df.isnull().sum()

In [ ]:
df = df.drop("customerID", axis=1)

In [ ]:
df.columns

In [ ]:
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

In [ ]:
df["Churn"].value_counts()

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [ ]:
X.head()

In [ ]:
X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
X_encoded.head()

In [ ]:
X_encoded.shape

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
df["TotalCharges"].isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()][
    ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]
]

In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [ ]:
df["TotalCharges"].isnull().sum()

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_encoded = pd.get_dummies(X, drop_first=True)

In [ ]:
print(X_encoded.isnull().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]

In [ ]:
y_prob[:10]

In [ ]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_prob)

print("ROC-AUC:", auc)

In [ ]:
X_train.dtypes

In [ ]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


In [ ]:
X_train.isnull().sum().sum()


In [ ]:
X_train.isnull().sum()[X_train.isnull().sum() > 0]

In [ ]:
median_totalcharges = X_train["TotalCharges"].median()

X_train["TotalCharges"] = X_train["TotalCharges"].fillna(median_totalcharges)
X_test["TotalCharges"] = X_test["TotalCharges"].fillna(median_totalcharges)

In [ ]:

feature_names = X_train.columns
coefficients = model.named_steps["logistic_regression"].coef_[0]


marketing_insights = pd.DataFrame({
    'Feature': feature_names,
    'Weight': coefficients
})
marketing_insights['Odds_Ratio'] = np.exp(marketing_insights['Weight'])


print(marketing_insights.sort_values(by='Weight', ascending=False))



In [ ]:

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score


X_cat = X.copy()


X_cat["TotalCharges"] = X_cat["TotalCharges"].fillna(X_cat["TotalCharges"].median())


categorical_features_indices = X_cat.select_dtypes(include=["object", "bool"]).columns.tolist()


X_train_cb, X_test_cb, y_train_cb, y_test_cb = train_test_split(
    X_cat,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


cb_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    random_seed=42,
    verbose=100 
)

cb_model.fit(
    X_train_cb, y_train_cb,
    cat_features=categorical_features_indices,
    eval_set=(X_test_cb, y_test_cb),
    early_stopping_rounds=50 
)


y_pred_cb = cb_model.predict(X_test_cb)
y_prob_cb = cb_model.predict_proba(X_test_cb)[:, 1]

print("\n=== CATBOOST PERFORMANCE REPORT ===")
print("Accuracy:", accuracy_score(y_test_cb, y_pred_cb))
print("\nClassification Report:")
print(classification_report(y_test_cb, y_pred_cb))
print("ROC-AUC:", roc_auc_score(y_test_cb, y_prob_cb))



In [ ]:


assert (X_test_cb.index == X_test.index).all(), "Index mismatch between test sets!"
assert (y_test_cb.index == y_test.index).all(), "Index mismatch between target labels!"


dashboard_data = pd.DataFrame({
    'Actual_Churn': y_test_cb,
    'LogReg_Prob': model.predict_proba(X_test)[:, 1],
    'CatBoost_Prob': y_prob_cb
}, index=X_test_cb.index)


print(dashboard_data.head())

In [ ]:
from sklearn.metrics import roc_auc_score

print("Logistic Regression AUC:",
      roc_auc_score(y_test_cb, dashboard_data["LogReg_Prob"]))

print("CatBoost AUC:",
      roc_auc_score(y_test_cb, dashboard_data["CatBoost_Prob"]))


In [ ]:
import pandas as pd
import os
assert X_test_cb.index.equals(X_test.index), "Test sets do not match!"
assert X_test_cb.index.equals(y_test_cb.index), "Target indices do not match!"

dashboard_data = X_test_cb.copy()

dashboard_data["Actual_Churn"] = y_test_cb


dashboard_data["LogReg_Prob"] = model.predict_proba(X_test)[:, 1]
dashboard_data["CatBoost_Prob"] = cb_model.predict_proba(X_test_cb)[:, 1]


def assign_risk(prob):
    if prob >= 0.70:
        return "High Risk"
    elif prob >= 0.30:
        return "Medium Risk"
    else:
        return "Low Risk"

dashboard_data["Risk_Segment"] = dashboard_data["CatBoost_Prob"].apply(assign_risk)


dashboard_data["Annual_Revenue"] = dashboard_data["MonthlyCharges"] * 12
dashboard_data["Expected_Revenue_At_Risk"] = dashboard_data["Annual_Revenue"] * dashboard_data["CatBoost_Prob"]


filename = "telco_churn_predictions_bi.csv"
dashboard_data.to_csv(filename, index=False)
print("CSV exported successfully!")
print("File saved at:", os.path.abspath(filename))
print("Dataset shape:", dashboard_data.shape)
display(dashboard_data.head())

In [ ]:
import os


def assign_risk(prob):
    if prob >= 0.70:
        return "High Risk"
    elif prob >= 0.30:
        return "Medium Risk"
    else:
        return "Low Risk"


dashboard_data["Risk_Segment"] = dashboard_data["CatBoost_Prob"].apply(assign_risk)


dashboard_data["Annual_Revenue"] = dashboard_data["MonthlyCharges"] * 12
dashboard_data["Expected_Revenue_At_Risk"] = dashboard_data["Annual_Revenue"] * dashboard_data["CatBoost_Prob"]


filename = "telco_churn_predictions_bi.csv"
dashboard_data.to_csv(filename, index=False)

print("CSV exported successfully!")
print("File saved at:", os.path.abspath(filename))
print("Dataset shape:", dashboard_data.shape)
display(dashboard_data.head())

In [ ]:
import os

print("File exists:", os.path.exists("telco_churn_predictions_bi.csv"))
print("Location:", os.path.abspath("telco_churn_predictions_bi.csv"))
print("Total customers:", len(dashboard_data))

In [ ]:
# Install requirements from the repository root before running: pip install -r requirements.txt


In [ ]:
import shap
import matplotlib.pyplot as plt


explainer = shap.TreeExplainer(cb_model)


shap_values = explainer(X_test_cb)

shap.plots.bar(shap_values, max_display=15)

plt.show()


In [ ]:
# SHAP beeswarm plot
shap.plots.beeswarm(shap_values, max_display=15)

In [ ]:

PROFIT_MARGIN = 0.30
EXPECTED_MONTHS = 12


clv_data = dashboard_data.copy()


clv_data["Estimated_CLV"] = (
    clv_data["MonthlyCharges"]
    * PROFIT_MARGIN
    * EXPECTED_MONTHS
)


clv_data["CLV_At_Risk"] = (
    clv_data["Estimated_CLV"]
    * clv_data["CatBoost_Prob"]
)

display(
    clv_data[
        [
            "MonthlyCharges",
            "CatBoost_Prob",
            "Risk_Segment",
            "Estimated_CLV",
            "CLV_At_Risk"
        ]
    ].head()
)

In [ ]:

priority_customers = clv_data.sort_values(
    by="CLV_At_Risk",
    ascending=False
)

display(
    priority_customers[
        [
            "Contract",
            "tenure",
            "MonthlyCharges",
            "CatBoost_Prob",
            "Risk_Segment",
            "Estimated_CLV",
            "CLV_At_Risk"
        ]
    ].head(10)
)

In [ ]:
import os

filename = "telco_churn_predictions_bi.csv"

clv_data.to_csv(filename, index=False)

print("Updated CSV exported successfully!")
print("File location:", os.path.abspath(filename))
print("Customers:", len(clv_data))
print("Columns:", len(clv_data.columns))

In [ ]:
import pandas as pd

df_check = pd.read_csv("telco_churn_predictions_bi.csv")

print(df_check.columns.tolist())

In [ ]:

def retention_strategy(row):
    if row["Risk_Segment"] == "High Risk":
        return "Personalized retention outreach"
    elif row["Risk_Segment"] == "Medium Risk":
        return "Engagement and service review"
    else:
        return "Loyalty and satisfaction monitoring"

clv_data["Retention_Strategy"] = clv_data.apply(
    retention_strategy, axis=1
)


clv_data["Retention_Priority"] = (
    clv_data["CLV_At_Risk"]
    .rank(ascending=False, method="dense")
    .astype(int)
)

display(
    clv_data[
        [
            "Contract",
            "tenure",
            "CatBoost_Prob",
            "Risk_Segment",
            "CLV_At_Risk",
            "Retention_Priority",
            "Retention_Strategy"
        ]
    ].sort_values("Retention_Priority").head(10)
)

In [ ]:
clv_data.to_csv(
    "telco_churn_predictions_bi.csv",
    index=False
)

print("Final customer retention dataset exported!")

In [ ]:
clv_data.to_csv(
    "telco_churn_predictions_final.csv",
    index=False
)

print("Final dataset exported!")
print("Customers:", len(clv_data))
print("Columns:", len(clv_data.columns))

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

y_pred_cb = cb_model.predict(X_test_cb)

print("CATBOOST MODEL EVALUATION")
print("-------------------------")

print("ROC-AUC:",
      roc_auc_score(y_test_cb, cb_model.predict_proba(X_test_cb)[:, 1]))

print("\nClassification Report:")
print(classification_report(y_test_cb, y_pred_cb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_cb, y_pred_cb))

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

threshold_results = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70]:
    predictions = (y_prob_cb >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_test_cb, predictions, zero_division=0),
        "Recall": recall_score(y_test_cb, predictions, zero_division=0),
        "F1": f1_score(y_test_cb, predictions, zero_division=0),
        "Customers_Flagged": predictions.sum()
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(3))